In [9]:
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime

# Load the datasets
m2_df = pd.read_csv('macro/fredData/m2.csv')
btc_df = pd.read_csv('micro/assetData/bitcoin.csv')

# Convert dates to datetime
m2_df['date'] = pd.to_datetime(m2_df['date'])
btc_df['date'] = pd.to_datetime(btc_df['date'])

# Filter M2 data to match Bitcoin's date range
m2_df = m2_df[m2_df['date'] >= btc_df['date'].min()]
m2_df = m2_df[m2_df['date'] <= btc_df['date'].max()]


m2_df['yoy'] = m2_df['m2'].pct_change(periods=12) * 100

# Calculate variations for Bitcoin (using price column)

btc_df['yoy'] = btc_df.iloc[:, 1].pct_change(periods=365) * 100

# Create dual-axis YoY comparison plot
fig_yoy = go.Figure()
fig_yoy.add_trace(go.Scatter(x=m2_df['date'], y=m2_df['yoy'], name='M2 YoY', line=dict(color='blue')))
fig_yoy.add_trace(go.Scatter(x=btc_df['date'], y=btc_df['yoy'], name='Bitcoin YoY', 
                            line=dict(color='orange'), yaxis='y2'))
fig_yoy.update_layout(
    title='Year-over-Year Variations: M2 vs Bitcoin',
    xaxis_title='Date',
    yaxis_title='M2 Variation (%)',
    yaxis2=dict(title='Bitcoin Variation (%)', overlaying='y', side='right'),
    legend=dict(x=1.1, y=1)
)


fig_yoy.show()

In [11]:
import requests

url = "https://api.db.nomics.world/v22/series/ISM/pmi?facets=1&format=json&limit=1000&observations=1"
response = requests.get(url)
data = response.json()
print(data)

{'_meta': {'args': {'align_periods': False, 'dataset_code': 'pmi', 'dimensions': {}, 'facets': True, 'format': 'json', 'limit': 1000, 'metadata': True, 'observations': True, 'offset': 0, 'provider_code': 'ISM', 'q': '', 'series_code': None}, 'version': '22.1.17'}, 'dataset': {'code': 'pmi', 'dimensions_codes_order': ['frequency'], 'dimensions_labels': {'frequency': 'Frequency'}, 'dimensions_values_labels': {'frequency': {'M': 'Monthly'}}, 'dir_hash': '281b1f57e08e66f2d02fc328fd53df175a005d2f', 'indexed_at': '2025-01-05T02:13:19.536Z', 'name': 'Manufacturing - PMI', 'nb_series': 1, 'next_release_at': '2025-02-03', 'provider_code': 'ISM', 'provider_name': 'Institute for Supply Management'}, 'errors': None, 'provider': {'code': 'ISM', 'indexed_at': '2025-02-03T03:04:34.394Z', 'name': 'Institute for Supply Management', 'region': 'US', 'slug': 'ism', 'terms_of_use': 'https://www.ismworld.org/footer/terms-of-use/', 'website': 'https://www.ismworld.org'}, 'series': {'docs': [{'@frequency': 'm

In [16]:
import requests
import pandas as pd

def parse_pmi_json(json_data):
    """
    Parses PMI data from the given JSON.
    """
    try:
        # 'json_data' is already a Python dictionary, so we can just extract from it
        series_data = json_data["series"]["docs"][0]

        periods = series_data["period"]  # List of time periods (monthly)
        values = series_data["value"]    # Corresponding PMI values

        # Create a DataFrame
        df = pd.DataFrame({
            "Date": pd.to_datetime(periods),
            "PMI": values
        })

        df.sort_values("Date", inplace=True)  # Ensure chronological order
        df.set_index("Date", inplace=True)    # Set Date as index
        return df
    except Exception as e:
        print(f"Error parsing JSON: {e}")
        return None

if __name__ == "__main__":
    url = "https://api.db.nomics.world/v22/series/ISM/pmi?facets=1&format=json&limit=1000&observations=1"
    response = requests.get(url)
    data = response.json()

    # Parse the JSON into a DataFrame
    df_pmi = parse_pmi_json(data)

    if df_pmi is not None:
        print(df_pmi)


             PMI
Date            
2020-05-01  43.1
2020-06-01  52.2
2020-07-01  53.7
2020-08-01  55.6
2020-09-01  55.7
2020-10-01  58.8
2020-11-01  57.7
2020-12-01  60.5
2021-01-01  58.7
2021-02-01  60.9
2021-03-01  63.7
2021-04-01  60.6
2021-05-01  61.6
2021-06-01  60.9
2021-07-01  59.9
2021-08-01  59.7
2021-09-01  60.5
2021-10-01  60.8
2021-11-01  60.6
2021-12-01  58.8
2022-01-01  57.6
2022-02-01  58.4
2022-03-01  57.0
2022-04-01  55.9
2022-05-01  56.1
2022-06-01  53.1
2022-07-01  52.7
2022-08-01  52.9
2022-09-01  51.0
2022-10-01  50.0
2022-11-01  49.0
2022-12-01  48.4
2023-01-01  47.4
2023-02-01  47.7
2023-03-01  46.5
2023-04-01  47.0
2023-05-01  46.6
2023-06-01  46.4
2023-07-01  46.5
2023-08-01  47.6
2023-09-01  48.6
2023-10-01  46.9
2023-11-01  46.6
2023-12-01  47.1
2024-01-01  49.1
2024-02-01  47.8
2024-03-01  50.3
2024-04-01  49.2
2024-05-01  48.7
2024-06-01  48.5
2024-07-01  46.8
2024-08-01  47.2
2024-09-01  47.2
2024-10-01  46.5
2024-11-01  48.4
2024-12-01  49.3


In [18]:
import quandl
import pandas as pd




In [21]:
quandl.ApiConfig.api_key = 'tEsTkEy123456789'

pmi_data = quandl.get("ISM/MAN_PMI")

print(pmi_data.head())

QuandlError: (Status 403) Something went wrong. Please try again. If you continue to have problems, please contact us at connect@quandl.com.